# TRAIN REID

In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' # Ẩn các log thừa của TensorFlow

In [2]:
!git clone https://github.com/JDAI-CV/fast-reid.git

fatal: destination path 'fast-reid' already exists and is not an empty directory.


In [3]:
%cd fast-reid

/kaggle/working/fast-reid


In [ ]:
!pip install -r docs/requirements.txt

In [ ]:
!pip install faiss-cpu

In [ ]:
!pip install --upgrade yacs

In [ ]:
!pip uninstall -y numpy
!pip install numpy==1.26.4

In [47]:
!cp -r /kaggle/input/my-vehicle-eval-2/my_vehicle_eval datasets

In [48]:
ls datasets

my_vehicle_eval/  README.md  vehicle_reid_dataset/


In [9]:
!mkdir checkpoints

In [ ]:
!wget https://github.com/JDAI-CV/fast-reid/releases/download/v0.1.1/veri_sbs_R50-ibn.pth -O pretrained/veri_sbs_R50-ibn.pth

In [17]:
ls pretrained

veri_sbs_R50-ibn.pth  veriwild_bot_R50-ibn.pth


In [13]:
# Chạy cell này để tạo file my_vehicle.py
import os

# Tạo thư mục chứa dataset script nếu chưa có
os.makedirs('fastreid/data/datasets', exist_ok=True)

path_to_file = '/kaggle/working/fast-reid/fastreid/data/datasets/my_vehicle.py'

content = """
import glob
import os.path as osp
import re
from .bases import ImageDataset
from ..datasets import DATASET_REGISTRY

@DATASET_REGISTRY.register()
class MyVehicleDataset(ImageDataset):
    dataset_dir = "vehicle_reid_dataset"
    dataset_name = "my_vehicle"

    def __init__(self, root='datasets', **kwargs):
        self.root = root
        self.dataset_dir = osp.join(self.root, self.dataset_dir)

        # Sử dụng đúng tên thư mục bạn đã chuẩn bị
        train_dir = osp.join(self.dataset_dir, 'bounding_box_train')
        query_dir = osp.join(self.dataset_dir, 'query')
        gallery_dir = osp.join(self.dataset_dir, 'bounding_box_test')

        self.check_before_run([train_dir, query_dir, gallery_dir])

        # Phân biệt is_train để xử lý nhãn (label)
        train = self.process_dir(train_dir, is_train=True)
        query = self.process_dir(query_dir, is_train=False)
        gallery = self.process_dir(gallery_dir, is_train=False)

        super(MyVehicleDataset, self).__init__(train, query, gallery, **kwargs)

    def process_dir(self, dir_path, is_train=True):
        img_paths = glob.glob(osp.join(dir_path, '*.jpg'))
        # Regex sửa lại để khớp chính xác: số_c số_phần còn lại
        pattern = re.compile(r'(\\d+)_c(\\d+)')

        data = []
        for img_path in img_paths:
            filename = osp.basename(img_path)
            match = pattern.search(filename)
            if not match: 
                continue

            pid, camid = map(int, match.groups())
            
            # Bỏ qua junk images (ID 0 hoặc -1 tùy quy ước của bạn)
            if pid <= 0 and is_train: 
                continue
            
            # CameraID trong FastReID nên bắt đầu từ 0
            camid -= 1 

            if is_train:
                # Ép kiểu string kèm tên dataset để đồng nhất với logic VeRi
                pid = self.dataset_name + "_" + str(pid)
                camid = self.dataset_name + "_" + str(camid)
            
            data.append((img_path, pid, camid))
        return data
"""

with open(path_to_file, 'w') as f:
    f.write(content)

# Quan trọng: Đăng ký dataset vào file __init__.py để FastReID nhận diện
# with open('fastreid/data/datasets/__init__.py', 'a') as f:
#     f.write("\nfrom .my_vehicle import MyVehicleDataset")

print("Đã tạo file my_vehicle.py và đăng ký dataset thành công!")

Đã tạo file my_vehicle.py và đăng ký dataset thành công!


In [28]:
os.makedirs('configs/MyVehicle', exist_ok=True)

config_content = """
_BASE_: "../Base-bagtricks.yml"

INPUT:
  SIZE_TRAIN: [256, 256]
  SIZE_TEST: [256, 256]
  REA:
    ENABLED: True
    PROB: 0.5 # Tăng nhẹ để mô hình bền bỉ hơn với dữ liệu camera thực tế

MODEL:
  WEIGHTS: "/kaggle/working/fast-reid/pretrained/veri_sbs_R50-ibn.pth"

  BACKBONE:
    NAME: "build_resnet_backbone"
    DEPTH: 50x
    WITH_IBN: True
    PRETRAIN: False

  HEADS:
    NAME: "EmbeddingHead"
    NUM_CLASSES: 0 
    POOL_LAYER: "GeneralizedMeanPooling" # GeM Pooling rất tốt cho Vehicle ReID
    WITH_BNNECK: True

  LOSSES:
    NAME: ("CrossEntropyLoss", "TripletLoss",)
    TRI:
      HARD_MINING: True
      MARGIN: 0.3

DATASETS:
  NAMES: ("MyVehicleDataset",)
  TESTS: ("MyVehicleDataset",)

SOLVER:
  OPT: "Adam"
  MAX_EPOCH: 60 # Với Vehicle ReID quy mô nhỏ, 60-80 epoch là đủ để hội tụ
  BASE_LR: 0.00035
  IMS_PER_BATCH: 16 # Điều chỉnh dựa trên VRAM của Kaggle (P100/T4)
  STEPS: [30, 50] # Giảm step tương ứng với MAX_EPOCH
  WARMUP_ITERS: 1000 # Cho mô hình thời gian khởi động lâu hơn một chút
  CHECKPOINT_PERIOD: 10
  AMP:
    ENABLED: False

OUTPUT_DIR: "/kaggle/working/logs_veri"
"""

with open('configs/MyVehicle/train_config.yml', 'w') as f:
    f.write(config_content)

print("Đã tạo file train_config.yml thành công!")

Đã tạo file train_config.yml thành công!


In [ ]:
# Gỡ cài đặt phiên bản hiện tại gây lỗi
!pip uninstall -y protobuf tensorboard

# Cài đặt lại các phiên bản ổn định và tương thích với nhau
!pip install protobuf==3.20.3 tensorboard==2.11.0

In [29]:
import os
import glob

def fix_collections_import(directory):
    # Tìm tất cả file .py trong thư mục fastreid
    files = glob.glob(os.path.join(directory, "**/*.py"), recursive=True)
    
    for file_path in files:
        with open(file_path, 'r') as f:
            content = f.read()
        
        # Trường hợp 1: Có cả Mapping và OrderedDict
        if "from collections import Mapping, OrderedDict" in content or "from collections.abc import Mapping, OrderedDict" in content:
            new_content = content.replace("from collections import Mapping, OrderedDict", 
                                        "from collections.abc import Mapping\nfrom collections import OrderedDict")
            new_content = new_content.replace("from collections.abc import Mapping, OrderedDict", 
                                            "from collections.abc import Mapping\nfrom collections import OrderedDict")
        
        # Trường hợp 2: Chỉ có Mapping
        elif "from collections import Mapping" in content:
            new_content = content.replace("from collections import Mapping", "from collections.abc import Mapping")
        
        else:
            continue
            
        with open(file_path, 'w') as f:
            f.write(new_content)
            print(f"Fixed: {file_path}")

# Thực hiện sửa lỗi
fix_collections_import('/kaggle/working/fast-reid/fastreid')
print("--- Hoàn tất sửa lỗi import! ---")

--- Hoàn tất sửa lỗi import! ---


In [30]:
!python tools/train_net.py --config-file configs/MyVehicle/train_config.yml 

E0000 00:00:1768751753.157585     450 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768751753.164028     450 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768751753.181155     450 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768751753.181183     450 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768751753.181187     450 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768751753.181190     450 computation_placer.cc:177] computation placer already registered. Please check linka

In [ ]:
# 1. Gỡ cài đặt các thư viện đang xung đột
!pip uninstall -y numpy scipy scikit-learn

# 2. Cài đặt lại phiên bản ổn định tương thích với FastReID và Kaggle
!pip install numpy==1.26.4 scipy scikit-learn

In [6]:
import os
import cv2
import torch
import numpy as np
from pathlib import Path
import sys
from PIL import Image

# Thiết lập đường dẫn để Python tìm thấy module fastreid
sys.path.append('/kaggle/working/fast-reid')

from fastreid.config import get_cfg
from fastreid.engine import DefaultPredictor
from fastreid.data.transforms import build_transforms

class EmbeddingExtractor:
    def __init__(self, config_file, weights_path, device="cuda"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        # 1. Khởi tạo Config
        self.cfg = get_cfg()
        self.cfg.merge_from_file(config_file)
        self.cfg.MODEL.DEVICE = self.device
        self.cfg.MODEL.WEIGHTS = weights_path
        
        # Lưu ý quan trọng: Phải đặt lại số class nếu model được train với số class khác mặc định
        # self.cfg.MODEL.HEADS.NUM_CLASSES = 294 
        
        self.cfg.freeze()

        # 2. Khởi tạo Predictor (Sử dụng wrapper của FastReID để đơn giản hóa)
        self.predictor = DefaultPredictor(self.cfg)
        
        # 3. Khởi tạo Transform chuẩn của FastReID
        self.transform = build_transforms(self.cfg, is_train=False)
        print(f"-> Đã nạp Model weights từ: {weights_path}")
        print(f"-> Thiết bị đang dùng: {self.device}")

    def extract_single_image(self, image_source):
        """
        Trích xuất từ đường dẫn hoặc numpy array (cv2 image)
        """
        if isinstance(image_source, (str, Path)):
            image = cv2.imread(str(image_source))
            if image is None: raise ValueError("Ảnh không tồn tại!")
        else:
            image = image_source

        # Tiền xử lý
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_pil = Image.fromarray(image_rgb)
        img_tensor = self.transform(image_pil).unsqueeze(0).to(self.device)

        # Trích xuất
        with torch.no_grad():
            # Lấy feature từ model
            # Lưu ý: feat thường có shape (1, 2048) với ResNet50
            feat = self.predictor.model(img_tensor)
        
        # Chuyển sang numpy và chuẩn hóa L2
        feat = feat.cpu().numpy().flatten()
        norm = np.linalg.norm(feat)
        if norm > 0:
            feat = feat / norm
            
        return feat

    def extract_from_crops(self, list_of_crops, method="mean"):
        """
        Dùng cho tracking: Tính embedding trung bình từ nhiều frame của cùng 1 xe
        """
        if not list_of_crops: return None
        
        all_feats = [self.extract_single_image(crop) for crop in list_of_crops]
        all_feats = np.array(all_feats)

        if method == "mean":
            final_feat = np.mean(all_feats, axis=0)
        else:
            final_feat = np.max(all_feats, axis=0)

        # Chuẩn hóa lại sau khi tính trung bình
        final_feat = final_feat / np.linalg.norm(final_feat)
        return final_feat

# --- CÁCH SỬ DỤNG TRÊN KAGGLE ---

# 1. Cấu hình đường dẫn
CFG_PATH = "/kaggle/working/fast-reid/configs/MyVehicle/train_config.yml"
WEIGHT_PATH = "/kaggle/working/logs/model_final.pth" # Hoặc model_best.pth

# 2. Khởi tạo extractor
extractor = EmbeddingExtractor(CFG_PATH, WEIGHT_PATH)

# 3. Test thử với một ảnh trong tập query
test_img_path = "/kaggle/working/fast-reid/datasets/vehicle_reid_dataset/query/0516_c3_f002605.jpg"
if os.path.exists(test_img_path):
    vector = extractor.extract_single_image(test_img_path)
    print(f"\nKích thước Vector: {vector.shape}")
    print(f"5 giá trị đầu tiên: {vector[:5]}")
else:
    print("Vui lòng kiểm tra lại đường dẫn ảnh test!")

-> Đã nạp Model weights từ: /kaggle/working/logs/model_final.pth
-> Thiết bị đang dùng: cuda

Kích thước Vector: (2048,)
5 giá trị đầu tiên: [ 0.00754255  0.04167599  0.01503585 -0.01673599 -0.07037923]


# TEST REID

In [51]:
path_to_file = 'fastreid/data/datasets/my_vehicle_eval.py'

content = """
import glob
import os.path as osp
import re

from .bases import ImageDataset
from ..datasets import DATASET_REGISTRY

@DATASET_REGISTRY.register()
class MyVehicleEval(ImageDataset):
    # Đường dẫn: datasets/my_vehicle_eval/
    dataset_dir = "my_vehicle_eval"
    dataset_name = "my_vehicle_eval"

    def __init__(self, root='datasets', **kwargs):
        self.root = root
        self.dataset_dir = osp.join(self.root, self.dataset_dir)

        query_dir = osp.join(self.dataset_dir, 'query')
        gallery_dir = osp.join(self.dataset_dir, 'bounding_box_test')

        # Kiểm tra sự tồn tại của thư mục query và gallery
        self.check_before_run([query_dir, gallery_dir])

        # Thực hiện xử lý dữ liệu (is_train=False cho cả hai)
        query = self.process_dir(query_dir, is_train=False)
        gallery = self.process_dir(gallery_dir, is_train=False)

        # Khởi tạo với tập train là danh sách rỗng để chỉ thực hiện test/eval
        super(MyVehicleEval, self).__init__([], query, gallery, **kwargs)

    def process_dir(self, dir_path, is_train=False):
        img_paths = glob.glob(osp.join(dir_path, '*.jpg'))
        # Regex khớp với định dạng ID_c(Số): ví dụ 0001_c1_f001.jpg
        pattern = re.compile(r'(\\d+)_c(\\d+)') 

        data = []
        for img_path in img_paths:
            filename = osp.basename(img_path)
            match = pattern.search(filename)
            if not match: 
                continue

            pid, camid = map(int, match.groups())
            
            # Junk ID thường là -1 hoặc 0, tùy thuộc vào cách bạn đánh nhãn
            if pid == -1: 
                continue 
            
            # FastReID yêu cầu camid bắt đầu từ 0 (c1 -> 0, c2 -> 1)
            camid -= 1 

            # Lưu ý: Vì đây là tập Eval, không cần thêm dataset_name vào pid như tập Train
            data.append((img_path, pid, camid))
            
        return data
"""

with open(path_to_file, 'w') as f:
    f.write(content)

# Quan trọng: Đăng ký dataset vào file __init__.py để FastReID nhận diện
# with open('fastreid/data/datasets/__init__.py', 'a') as f:
#     f.write("\nfrom .my_vehicle_eval import MyVehicleEval")

print("Đã tạo file my_vehicle_eval.py và đăng ký dataset thành công!")


Đã tạo file my_vehicle_eval.py và đăng ký dataset thành công!


In [49]:
os.makedirs('configs/MyVehicle', exist_ok=True)

config_content = """
_BASE_: "../Base-bagtricks.yml"

INPUT:
  SIZE_TRAIN: [256, 256]
  SIZE_TEST: [256, 256]

MODEL:
  # Đường dẫn tuyệt đối tới model bạn đã train xong
  WEIGHTS: "/kaggle/working/fast-reid/pretrained/veri_sbs_R50-ibn.pth"
  
  BACKBONE:
    NAME: "build_resnet_backbone"
    DEPTH: 50x
    WITH_IBN: True
    PRETRAIN: False

  HEADS:
    NAME: "EmbeddingHead"
    # Để 0 để tự động khớp với weight đã nạp và dữ liệu test
    NUM_CLASSES: 0 
    POOL_LAYER: "GeneralizedMeanPooling" 
    WITH_BNNECK: True

DATASETS:
  # Tên dataset phải khớp với class MyVehicleEval bạn đã đăng ký
  NAMES: ("MyVehicleEval",)
  TESTS: ("MyVehicleEval",)

SOLVER:
  # Các thông số này không quan trọng khi chạy --eval-only nhưng nên để giá trị an toàn
  IMS_PER_BATCH: 64 

OUTPUT_DIR: "/kaggle/working/logs/eval_results"
"""

# with open('configs/MyVehicle/test_config.yml', 'w') as f:
#     f.write(config_content)

print("Đã tạo file test_config.yml thành công!")

Đã tạo file test_config.yml thành công!


In [20]:
!cp -r /kaggle/input/my-vehicle-eval/my_vehicle_eval datasets

In [35]:
ls datasets

my_vehicle_eval/  README.md  vehicle_reid_dataset/


In [36]:
# Veri đã train trên tập 1 --> 2, 3.
!python tools/train_net.py --config-file configs/MyVehicle/test_config.yml --eval-only

E0000 00:00:1768761018.329131     611 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768761018.335532     611 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768761018.352514     611 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768761018.352543     611 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768761018.352546     611 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768761018.352550     611 computation_placer.cc:177] computation placer already registered. Please check linka